# ReneWind — Wind Turbine Generator Failure Prediction (Full Code)

**UT Austin PGP-style neural network project** | Predictive maintenance using sensor data

## Problem Statement

### Business Context

Renewable energy sources play an increasingly important role in the global energy mix, as efforts to reduce the environmental impact of energy production increase.

Wind energy is among the most mature renewable technologies worldwide. The U.S. Department of Energy recommends **predictive maintenance**: using sensor data and analytics to predict degradation before catastrophic failure, lowering operations and maintenance (O&M) cost.

Sensors on wind turbines capture environmental readings (temperature, humidity, wind speed, etc.) and mechanical signals from components (gearbox, tower, blades, brake, etc.).

### Objective

**ReneWind** collects confidential, ciphered sensor data to predict **generator failure** (`Target = 1`). The training set has **20,000** rows and the held-out test set has **5,000** rows across **40** predictors.

We build and tune **neural network classifiers** so failures are caught early (repair) instead of missed (replacement).

| Outcome | Business impact |
| --- | --- |
| **TP** — failure predicted correctly | Repair cost (preferred) |
| **FN** — failure missed | Replacement cost (highest) |
| **FP** — false alarm | Inspection cost (lowest) |

**Cost hierarchy:** Replacement > Repair > Inspection. Minimizing **false negatives** is the top priority.

### Data Description

- `Train.csv` — training and hyperparameter tuning only
- `Test.csv` — **final** evaluation only (no tuning on test data)

Both files contain 40 predictors (`V1`–`V40`) and binary `Target` (1 = failure, 0 = no failure).

## 1 — Import Libraries

In [ ]:
# Standard library
import os
import time
import warnings

# Data and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# Scikit-learn
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import SGD, Adam

# Reproducibility and clean submission output
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
SEED = 1
np.random.seed(SEED)
tf.random.set_seed(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_style("whitegrid")


## 2 — Load Data

In [ ]:
DATA_DIR = "data"
train_path = os.path.join(DATA_DIR, "Train.csv")
test_path = os.path.join(DATA_DIR, "Test.csv")

data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

data_copy = data.copy()
test_copy = test_data.copy()


## 3 — Exploratory Data Analysis

### 3.1 Data Overview

In [ ]:
data.head()

In [ ]:
test_data.head()

In [ ]:
print('Train shape:', data.shape)
print('Test shape:', test_data.shape)

In [ ]:
data.info()

In [ ]:
data.describe(include='all').T

In [ ]:
print('Train duplicates:', data.duplicated().sum())
print('Test duplicates:', test_data.duplicated().sum())

In [ ]:
missing_train = (data.isnull().sum() / len(data) * 100).round(2)
missing_test = (test_data.isnull().sum() / len(test_data) * 100).round(2)
print('Missing % (train):\n', missing_train[missing_train > 0])
print('Missing % (test):\n', missing_test[missing_test > 0])


**Observation:** Training data has 20,000 rows × 41 columns; test has 5,000 rows. Only **V1** and **V2** show small missingness (<1%). No duplicate rows. All predictors are numeric (ciphered sensor channels).

### 3.2 Univariate Analysis

In [ ]:
def histogram_boxplot(df, feature, figsize=(10, 5)):
    """Combined histogram and boxplot for one feature."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    sns.histplot(data=df, x=feature, ax=axes[0], kde=True)
    axes[0].set_title(f'Distribution — {feature}')
    sns.boxplot(data=df, y=feature, ax=axes[1])
    axes[1].set_title(f'Boxplot — {feature}')
    plt.tight_layout()
    plt.show()


In [ ]:
# Target distribution (class imbalance is critical for maintenance cost)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=data, x='Target', ax=ax[0])
ax[0].set_title('Target — Train')
sns.countplot(data=test_data, x='Target', ax=ax[1])
ax[1].set_title('Target — Test')
plt.tight_layout()
plt.show()

print('Train counts:\n', data['Target'].value_counts())
print('Train proportions:\n', data['Target'].value_counts(normalize=True).round(4))


In [ ]:
predictor_cols = [c for c in data.columns if c != 'Target']
# Univariate plots for all 40 sensor predictors
for feature in predictor_cols:
    histogram_boxplot(data, feature, figsize=(9, 4))


**Observation:** Failures (`Target = 1`) are the **minority class** in both train and test, with similar proportions. Most sensor channels are roughly symmetric but some show skew and outliers (expected for environmental and mechanical signals).

### 3.3 Bivariate Analysis

In [ ]:
predictor_cols = [c for c in data.columns if c != 'Target']
corr_with_target = data[predictor_cols + ['Target']].corr()['Target'].drop('Target').sort_values(key=abs, ascending=False)

plt.figure(figsize=(8, 10))
sns.barplot(x=corr_with_target.head(15).values, y=corr_with_target.head(15).index, palette='viridis')
plt.title('Top 15 |Correlation| with Target (Train)')
plt.xlabel('Pearson correlation')
plt.tight_layout()
plt.show()


In [ ]:
top5 = corr_with_target.abs().sort_values(ascending=False).head(5).index.tolist()
fig, axes = plt.subplots(1, len(top5), figsize=(14, 4))
for ax, feat in zip(axes, top5):
    sns.boxplot(data=data, x='Target', y=feat, ax=ax)
    ax.set_title(feat)
plt.suptitle('Top correlated sensors vs. failure', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Pairwise correlation among top sensors + target
sns.heatmap(data[top5 + ['Target']].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Bivariate correlation — strongest sensors')
plt.tight_layout()
plt.show()


**Observation:** Several sensors (e.g., among the top correlated features) separate failure vs. non-failure distributions, supporting a nonlinear classifier. No target leakage is present in the feature set.

## 4 — Data Preprocessing

> Imputation and scaling are fit **only on the training split** after `train_test_split` to prevent **data leakage**.

In [ ]:
X = data.drop(columns=['Target'])
y = data['Target']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

X_test = test_data.drop(columns=['Target'])
y_test = test_data['Target']

print('Train / val / test shapes:', X_train.shape, X_val.shape, X_test.shape)
print('Class balance (train):', y_train.value_counts(normalize=True).round(4).to_dict())


In [ ]:
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
X_val = pd.DataFrame(imputer.transform(X_val), columns=X.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

y_train = y_train.to_numpy()
y_val = y_val.to_numpy()
y_test = y_test.to_numpy()

print('Missing after imputation (train):', np.isnan(X_train).sum())


**Observation:** V1/V2 missing values are imputed with training-set medians. Features are standardized using training statistics only. Class imbalance persists (~11% failures) and will be handled with **class weights** in most models.

## 5 — Model Building

### 5.1 Evaluation metric (rationale)

ReneWind's objective is to **minimize total maintenance spend**, not to maximize accuracy in isolation. Predictions map to dollars as follows:

| Outcome | Cost |
| --- | --- |
| TP (failure caught) | $15,000 repair |
| FN (failure missed) | $40,000 replacement |
| FP (false alarm) | $5,000 inspection |

**Maintenance cost** (validation / test):

`Maintenance cost = TP × $15,000 + FN × $40,000 + FP × $5,000`

**Cost efficiency ratio** (primary selection metric on validation):

`Cost efficiency = (TP + FN) × $15,000 ÷ Maintenance cost`

- Numerator = theoretical minimum spend if every true failure were only repaired (no replacements, no extra inspections).
- Denominator = actual spend implied by the confusion matrix.
- Range **0–1**; **higher is better** (1.0 = perfectly cost-efficient).

| Metric | Role |
| --- | --- |
| **Cost efficiency** | **Primary** — select the model/threshold that maximizes this on the **validation** set |
| **Maintenance cost ($)** | **Primary tie-breaker** — prefer lower dollar total when efficiency is similar |
| **Recall (failure class)** | **Secondary** — diagnostic for missed failures (FN); reported for operations |
| **Precision / F1** | Supporting — explains false-alarm (inspection) tradeoffs |

During training we still log **recall** (Keras metric) and use **class weights** so the network learns the rare failure class; **final champion selection uses validation dollars**, then a **probability threshold** is tuned on validation to further reduce maintenance cost.

In [ ]:
REPAIR_COST = 15_000
REPLACEMENT_COST = 40_000
INSPECTION_COST = 5_000

def maintenance_cost(cm):
    tn, fp, fn, tp = cm.ravel()
    return tp * REPAIR_COST + fn * REPLACEMENT_COST + fp * INSPECTION_COST

def cost_efficiency(cm):
    tn, fp, fn, tp = cm.ravel()
    minimum = (tp + fn) * REPAIR_COST
    actual = maintenance_cost(cm)
    return minimum / actual if actual else 0.0

def model_performance(model, X, y, threshold=0.5, label=''):
    prob = model.predict(X, verbose=0).ravel()
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y, pred)
    df_perf = pd.DataFrame({
        'Accuracy': [accuracy_score(y, pred)],
        'Recall': [recall_score(y, pred, zero_division=0)],
        'Precision': [precision_score(y, pred, zero_division=0)],
        'F1': [f1_score(y, pred, zero_division=0)],
        'Maintenance_Cost': [maintenance_cost(cm)],
        'Cost_Efficiency': [cost_efficiency(cm)],
    }, index=[label or 'model'])
    return df_perf, cm

def plot_history(history, metric_name, title):
    # Keras may log metrics as 'Recall' / 'val_Recall'
    keys = list(history.history.keys())
    train_key = next((k for k in keys if metric_name.lower() in k.lower() and not k.startswith('val_')), keys[0])
    val_key = next((k for k in keys if metric_name.lower() in k.lower() and k.startswith('val_')), f'val_{train_key}')
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history.history[train_key], label='Train')
    ax.plot(history.history[val_key], label='Validation')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(metric_name.capitalize())
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

def show_confusion(cm, title):
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()

# Class weights for imbalance
counts = np.bincount(y_train.astype(int))
cw_dict = {i: len(y_train) / counts[i] for i in range(len(counts))}
cw_dict


### 5.2 Helper to train and evaluate each architecture

In [ ]:
EPOCHS = 25
BATCH_SIZE = 32

results_train = {}
results_val = {}
results_test = {}
histories = {}

def train_and_evaluate(name, model, use_class_weight=True, config_note=''):
    tf.keras.backend.clear_session()
    start = time.time()
    fit_kwargs = dict(
        x=X_train, y=y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        verbose=0,
    )
    if use_class_weight:
        fit_kwargs['class_weight'] = cw_dict
    history = model.fit(**fit_kwargs)
    elapsed = time.time() - start

    train_perf, _ = model_performance(model, X_train, y_train, label=name)
    val_perf, val_cm = model_performance(model, X_val, y_val, label=name)
    test_perf, test_cm = model_performance(model, X_test, y_test, label=name)

    results_train[name] = train_perf
    results_val[name] = val_perf
    results_test[name] = test_perf
    histories[name] = history

    plot_history(history, 'loss', f'{name} — Loss\n{config_note}')
    plot_history(history, 'recall', f'{name} — Recall\n{config_note}')
    show_confusion(val_cm, f'{name} — Validation confusion matrix')
    show_confusion(test_cm, f'{name} — Test confusion matrix (held-out)')

    val_cost = int(val_perf['Maintenance_Cost'].iloc[0])
    val_eff = val_perf['Cost_Efficiency'].iloc[0]
    print(f'Training time: {elapsed:.1f}s')
    print(f'Validation maintenance cost: ${val_cost:,} | Cost efficiency: {val_eff:.4f}')
    display(train_perf)
    display(val_perf)
    return model, val_perf


### 5.3 Model 0 — Baseline neural network (SGD + class weights)

Single hidden layer (7 units, ReLU), sigmoid output, **SGD** optimizer — per project requirements.

In [ ]:
model_0 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(7, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model_0.compile(loss='binary_crossentropy', optimizer=SGD(), metrics=['Recall'])
model_0, _ = train_and_evaluate(
    'Model 0 — Baseline',
    model_0,
    config_note='1×7 ReLU, SGD, class weights, 25 epochs',
)


**Observation (Model 0):** Baseline **SGD** network. Compare **validation maintenance cost** and **cost efficiency** against later models — this is the dollar-based benchmark required by the project.

### 5.4 Model 1 — More hidden layers (2 layers)

**Technique:** additional hidden layer (14 → 7 neurons).

In [ ]:
model_1 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(14, activation='relu'),
    Dense(7, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model_1.compile(loss='binary_crossentropy', optimizer=SGD(), metrics=['Recall'])
model_1, _ = train_and_evaluate(
    'Model 1 — 2 hidden layers',
    model_1,
    config_note='14→7 ReLU, SGD, class weights',
)


**Observation (Model 1):** Two hidden layers increase capacity. If validation **cost efficiency** improves vs. Model 0, depth is helping the business KPI, not just training recall.

### 5.5 Model 2 — More hidden layers (3 layers)

**Technique:** third hidden layer for additional nonlinearity.

In [ ]:
model_2 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(21, activation='relu'),
    Dense(14, activation='relu'),
    Dense(7, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model_2.compile(loss='binary_crossentropy', optimizer=SGD(), metrics=['Recall'])
model_2, _ = train_and_evaluate(
    'Model 2 — 3 hidden layers',
    model_2,
    config_note='21→14→7 ReLU, SGD, class weights',
)


**Observation (Model 2):** Three hidden layers — check whether extra depth lowers validation **maintenance cost** or hurts generalization (efficiency drops vs. Model 1).

### 5.6 Model 3 — Adam optimizer

**Technique:** replace SGD with **Adam** (baseline depth).

In [ ]:
model_3 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(7, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model_3.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['Recall'])
model_3, _ = train_and_evaluate(
    'Model 3 — Adam optimizer',
    model_3,
    config_note='1×7 ReLU, Adam, class weights',
)


**Observation (Model 3):** **Adam** optimizer variant. Compare validation **cost efficiency** to SGD models — faster convergence only matters if dollars on validation improve.

### 5.7 Model 4 — Dropout regularization

**Technique:** **Dropout** (0.3) after hidden layers with 2-layer architecture.

In [ ]:
model_4 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(14, activation='relu'),
    Dropout(0.3),
    Dense(7, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid'),
])
model_4.compile(loss='binary_crossentropy', optimizer=SGD(), metrics=['Recall'])
model_4, _ = train_and_evaluate(
    'Model 4 — Dropout',
    model_4,
    config_note='14→7 + Dropout 0.3, SGD, class weights',
)


**Observation (Model 4):** **Dropout** regularization. Prefer this model if validation **maintenance cost** falls while recall stays acceptable (fewer costly FN replacements).

### 5.8 Model 5 — Without class weights

**Technique:** remove **class weights** to measure impact on minority (failure) class.

In [ ]:
model_5 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(7, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model_5.compile(loss='binary_crossentropy', optimizer=SGD(), metrics=['Recall'])
model_5, _ = train_and_evaluate(
    'Model 5 — No class weights',
    model_5,
    use_class_weight=False,
    config_note='1×7 ReLU, SGD, NO class weights',
)


**Observation (Model 5):** No **class weights** — expect higher validation **maintenance cost** (more FN → replacements). Confirms weighted training is required for dollar-optimal maintenance.

### 5.9 Model 6 — Combined best practices (Adam + depth + dropout + class weights)

**Technique:** combine **Adam**, **extra hidden layer**, **dropout**, and **class weights**.

In [ ]:
model_6 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(14, activation='relu'),
    Dropout(0.2),
    Dense(7, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid'),
])
model_6.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0005), metrics=['Recall'])
model_6, _ = train_and_evaluate(
    'Model 6 — Combined',
    model_6,
    config_note='14→7, Dropout 0.2, Adam, class weights',
)


**Observation (Model 6):** Combined **Adam + dropout + depth**. Strong candidate if it achieves the best validation **cost efficiency** among tunable architectures.

### 5.10 Model 7 — Adam + three hidden layers + dropout

**Technique:** deeper **Adam** network with dropout (sixth+ architecture variant).

In [ ]:
model_7 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(21, activation='relu'),
    Dropout(0.25),
    Dense(14, activation='relu'),
    Dropout(0.25),
    Dense(7, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model_7.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0005), metrics=['Recall'])
model_7, _ = train_and_evaluate(
    'Model 7 — Deep Adam + dropout',
    model_7,
    config_note='21→14→7, Dropout, Adam, class weights',
)


**Observation (Model 7):** Deepest **Adam + dropout** stack. Select only if validation **maintenance cost** is lowest or efficiency is highest vs. simpler models (avoid overfit that increases FN/FP dollars).

## 6 — Model Comparison and Final Selection (dollar-based)

In [ ]:
train_df = pd.concat(results_train.values(), axis=0)
val_df = pd.concat(results_val.values(), axis=0)
test_df = pd.concat(results_test.values(), axis=0)

print('=== Training metrics ===')
display(train_df)

print('=== Validation metrics — sorted by cost efficiency (primary KPI) ===')
val_sorted = val_df.sort_values(
    ['Cost_Efficiency', 'Maintenance_Cost'],
    ascending=[False, True],
)
display(val_sorted)

# Visual: validation maintenance cost by model
plot_df = val_sorted.reset_index().rename(columns={'index': 'Model'})
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=plot_df, x='Maintenance_Cost', y='Model', ax=axes[0], color='steelblue')
axes[0].set_title('Validation maintenance cost (lower is better)')
axes[0].set_xlabel('USD')
sns.barplot(data=plot_df, x='Cost_Efficiency', y='Model', ax=axes[1], color='seagreen')
axes[1].set_title('Validation cost efficiency (higher is better)')
axes[1].set_xlim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:
# --- Step 1: Pick architecture by validation dollars (not recall) ---
best_row = val_df.sort_values(
    ['Cost_Efficiency', 'Maintenance_Cost', 'Recall'],
    ascending=[False, True, False],
).iloc[0]
best_name = best_row.name
print('Selected architecture (validation cost efficiency):', best_name)
display(best_row.to_frame().T)

model_lookup = {
    'Model 0 — Baseline': model_0,
    'Model 1 — 2 hidden layers': model_1,
    'Model 2 — 3 hidden layers': model_2,
    'Model 3 — Adam optimizer': model_3,
    'Model 4 — Dropout': model_4,
    'Model 5 — No class weights': model_5,
    'Model 6 — Combined': model_6,
    'Model 7 — Deep Adam + dropout': model_7,
}
best_model = model_lookup[best_name]


### 6.1 Threshold tuning on validation (maintenance cost)

Default probability cutoff is 0.5. For O&M, we tune the cutoff on the **validation set only** so predicted maintenance **dollars** are minimized (equivalently, cost efficiency is maximized).

In [ ]:
threshold_grid = np.round(np.arange(0.10, 0.91, 0.05), 2)
threshold_rows = []

for t in threshold_grid:
    perf, cm = model_performance(best_model, X_val, y_val, threshold=float(t), label=f't={t}')
    threshold_rows.append({
        'threshold': t,
        'Maintenance_Cost': perf['Maintenance_Cost'].iloc[0],
        'Cost_Efficiency': perf['Cost_Efficiency'].iloc[0],
        'Recall': perf['Recall'].iloc[0],
        'FN': int(cm[1, 0]),
        'FP': int(cm[0, 1]),
        'TP': int(cm[1, 1]),
    })

threshold_df = pd.DataFrame(threshold_rows).sort_values(
    ['Cost_Efficiency', 'Maintenance_Cost'], ascending=[False, True]
)
display(threshold_df.head(10))

best_threshold_row = threshold_df.iloc[0]
BEST_THRESHOLD = float(best_threshold_row['threshold'])
print(f'Optimal validation threshold: {BEST_THRESHOLD}')
print(f"Validation maintenance cost at best threshold: ${int(best_threshold_row['Maintenance_Cost']):,}")
print(f"Validation cost efficiency: {best_threshold_row['Cost_Efficiency']:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(threshold_df['threshold'], threshold_df['Maintenance_Cost'], marker='o')
plt.xlabel('Probability threshold')
plt.ylabel('Validation maintenance cost (USD)')
plt.title(f'{best_name} — cost vs. threshold')
plt.tight_layout()
plt.show()


In [ ]:
# --- Step 2: One-time held-out test evaluation at dollar-optimal threshold ---
val_final_perf, val_final_cm = model_performance(
    best_model, X_val, y_val, threshold=BEST_THRESHOLD, label=best_name + ' (val, tuned)'
)
final_test_perf, final_cm = model_performance(
    best_model, X_test, y_test, threshold=BEST_THRESHOLD, label=best_name + ' (test)'
)

print('=== Final validation performance (tuned threshold) ===')
display(val_final_perf)
show_confusion(val_final_cm, f'Final — {best_name} (Validation, t={BEST_THRESHOLD})')

print('=== Final test performance (held-out; evaluate once) ===')
display(final_test_perf)
show_confusion(final_cm, f'Final — {best_name} (Test, t={BEST_THRESHOLD})')

tn, fp, fn, tp = final_cm.ravel()
print('Test confusion counts — TP:', tp, 'FN:', fn, 'FP:', fp, 'TN:', tn)
print('Test maintenance cost: $' + f"{int(final_test_perf['Maintenance_Cost'].iloc[0]):,}")
print('Test cost efficiency:', round(final_test_perf['Cost_Efficiency'].iloc[0], 4))


**Final model reasoning:**

1. **Architecture** — chosen by highest **validation cost efficiency**, then lowest **validation maintenance cost**, then recall as a tie-breaker.
2. **Threshold** — tuned on validation only to minimize maintenance dollars (same cost function as the business brief).
3. **Test set** — scored **once** at the tuned threshold; no test-set tuning (no leakage).

This aligns model selection with ReneWind's stated goal: **reduce O&M spend**, not optimize a surrogate metric in isolation.

## 7 — Actionable Insights and Business Recommendations

### Key takeaways

1. **Dollar-based selection:** The production model is the architecture with the best **validation cost efficiency**, with decision threshold tuned to minimize **validation maintenance cost** — directly aligned with the project objective.
2. **Class weights are economically necessary:** Removing weights (Model 5) increases FN-driven **replacement** spend; weighted training is required for cost-optimal predictions.
3. **Recall is a diagnostic, not the KPI:** High recall helps, but the right threshold balances **$40k replacements** vs. **$5k inspections**; report both recall and maintenance cost to operations.
4. **Depth, Adam, and dropout:** Architectures that improve **validation cost efficiency** (not just training recall) should be favored for deployment.

### Business recommendations

1. **Deploy with the tuned probability cutoff** — use the validation-derived threshold in SCADA alerting so dispatch triggers inspections/repairs when expected cost is minimized.
2. **Executive dashboard in dollars** — monthly TP/FN/FP counts translated to maintenance cost and cost efficiency vs. a "repair-only" baseline.
3. **Retrain quarterly** — refresh weights and **re-tune threshold** on recent validation folds; sensor drift changes both predictions and optimal cutoff.
4. **Pilot on high-correlation turbines** — prioritize sites where top EDA sensors are stable and failure labels are well-logged.

### Expected benefits

- Lower total generator O&M cost (fewer replacements, right-sized inspections)
- Planned repairs instead of emergency replacements
- Clear ROI narrative for ReneWind leadership (cost efficiency trending toward 1.0)

---

*End of notebook — export to HTML for submission.*
